In [1]:
# Install required packages.
# Note: Using `%pip` and `-U` ensures proper upgrades in the active Colab kernel.
%pip install -U langchain "langchain[google-genai]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 839.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 11.6 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.0
    Uninstalling langchain-core-1.6.0:
      Successfully uninstalled langchain-core-1.6.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled googl

------------------------------
## Método C (Estándar de Producción): Perfiles Dinámicos con Middleware

Mientras que los Métodos A y B son excelentes para configuraciones estáticas, las aplicaciones en producción a menudo requieren que la **misma instancia del agente** se comporte de manera diferente según el usuario o la tarea (ej. un perfil "creativo" para lluvia de ideas vs. un perfil "estricto" para extracción de datos).

Reconstruir el objeto del agente o del modelo en cada petición es ineficiente y viola el **Principio de Abierto/Cerrado** (SOLID).

### La Solución: Contexto en Tiempo de Ejecución y Middleware
La arquitectura moderna de LangChain nos permite interceptar el flujo de ejecución. Podemos pasar un diccionario `context` durante el `.invoke()`, y un **middleware** personalizado lo leerá y vinculará dinámicamente los hiperparámetros al modelo *justo milisegundos antes de la inferencia*, sin alterar el agente base en memoria.

### Beneficios Clave:
1. **Eficiencia de Memoria**: El agente se instancia una sola vez.
2. **Adaptación Dinámica**: Cambia la temperatura, tokens o penalizaciones por cada petición.
3. **Abstracción del Proveedor**: El middleware puede mapear nombres de perfiles genéricos a parámetros específicos del proveedor (ej. `max_output_tokens` para Gemini).

In [2]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware # Importamos la clase base oficial
from langchain.chat_models import init_chat_model
from google.colab import userdata

# 1. Configuración de la API Key
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# 2. Definición del Middleware (El "Gestor de Perfiles")
class DynamicProfileMiddleware(AgentMiddleware):
    """
    Middleware that intercepts the runtime context to inject hyperparameter profiles
    directly into the model call at runtime.
    """
    def wrap_model_call(self, runtime, handler):
        """
        Intercepts the model call. We read the context and bind parameters
        before the handler executes the actual LLM call.
        """
        # Recuperamos el contexto dinámico enviado en el .invoke()
        context = getattr(runtime, 'context', {}) or {}
        profile_options = context.get("generation_config", {})

        # Si el usuario especificó parámetros, los vinculamos al modelo sobre la marcha
        if profile_options:
            # .bind() creates a temporary copy of the model with these specific parameters
            runtime.model = runtime.model.bind(**profile_options)

        # Continue the execution chain
        return handler(runtime)

# 3. Inicializamos el agente UNA SOLA VEZ con el middleware registrado
# Usamos init_chat_model para máxima flexibilidad
base_model = init_chat_model("google_genai:gemini-3.6-flash")

agent = create_agent(
    model=base_model,
    system_prompt="You are a versatile AI assistant. Adapt your style to the requested profile.",
    middleware=[DynamicProfileMiddleware()]# El arnés ahora procesa perfiles dinámicos
)

# 4. EJECUCIÓN: PERFIL 1 (Creativo)
creative_profile = {"temperature": 0.9, "max_output_tokens": 100}

print(" Running with Creative Profile (Temperature 0.9)...")
creative_response = agent.invoke(
    {"messages": [{"role": "user", "content": "Write a 1-sentence sci-fi plot twist."}]},
    context={"generation_config": creative_profile} # Inyectando el perfil
)
print(f"AI 🤖: {creative_response['messages'][-1].content[0].get('text', '')}\n")

# 5. EJECUCIÓN: PERFIL 2 (Preciso/Estricto)
precise_profile = {"temperature": 0.1, "max_output_tokens": 50}

print(" Running with Precise Profile (Temperature 0.1)...")
precise_response = agent.invoke(
    {"messages": [{"role": "user", "content": "Write a 1-sentence sci-fi plot twist."}]},
    context={"generation_config": precise_profile}  # Inyectando un perfil diferente
)
print(f"AI 🤖: {precise_response['messages'][-1].content[0].get('text', '')}")

 Running with Creative Profile (Temperature 0.9)...
AI 🤖: After centuries of trying to breach the impenetrable energy field surrounding our solar system, we finally succeeded, only to intercept a galaxy-wide emergency broadcast announcing that the quarantine on Earth had failed.

 Running with Precise Profile (Temperature 0.1)...
AI 🤖: After a grueling two-hundred-year journey to reach humanity's new home, the colony ship's doors opened to reveal a sprawling metropolis, built by humans who had invented faster-than-light travel decades after the ship had launched.
